In [0]:
from pyspark.sql.functions import current_timestamp, col
from pathlib import Path

# Verbose mode
VERBOSE = True

# Define source paths and files/tables used
SOURCE_PATH = "/Volumes/workspace/mimic_data/s3_files/mimic-iv-dataset-demo-2.2-folder/hosp"
csv_file_names = ["diagnoses_icd", "d_icd_diagnoses", "drgcodes", "d_hcpcs", "hcpcsevents"]

# Define Target catalog and schema for bronze layer
TARGET_CATALOG = "workspace"
BRONZE_SCHEMA = "mimic_iv_dataset_bronze"

# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{BRONZE_SCHEMA}")

# Display details only if verbose is activated
if VERBOSE:
    print(f"Bronze layer ingestion: {SOURCE_PATH}")
    print(f"Target: {TARGET_CATALOG}.{BRONZE_SCHEMA}")
    print("-----")

# Ingesting the datasets (csv files)
for csv_file_name in csv_file_names:
    # Display details only if verbose is activated
    if VERBOSE:
        print(f"Processing: {csv_file_name}.csv")

    try:
        # Get file path and targeted table
        path = f"{SOURCE_PATH}/{csv_file_name}.csv"
        target_delta_table = f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{csv_file_name}"

        # Read csv file into dataframe
        df = (spark.read.format("csv")
                        .option("header", True)
                        .option("inferSchema", True)
                        .load(path))
        
        # Include Lineage in dataframe
        df = (df.withColumn("_ingestion_timestamp", current_timestamp())
                .withColumn("_source_file", col("_metadata.file_path")))
        
        # Write dataframe to delta table and store it in schema
        (df.write.format("delta")
                .mode("overwrite")
                .option("overwriteSchema", True)
                .saveAsTable(target_delta_table))
        
        # Display details only if verbose is activated
        if VERBOSE:
            print(f"Completed. Dataset details: {df.count()} data {len(df.columns)} features")
            print("")

    except Exception as e:
        print(f"Error processing file {csv_file_name}: {e}")

if VERBOSE:
    print("-----")
    print("Bronze Layer Ingestion successful")
    print(f"Created tables under {TARGET_CATALOG}.{BRONZE_SCHEMA}:")
    for csv_file_name in csv_file_names:
        print(f"\t- {csv_file_name}")

In [0]:
# Verify all delta tables in bronze layer exists through displaying samples
for csv_file_name in csv_file_names:
    print(f"Delta table: {csv_file_name}")
    
    df = spark.sql(f"SELECT * FROM workspace.mimic_iv_dataset_bronze.{csv_file_name} LIMIT 5")
    display(df)

    print("")

In [0]:
%sql

-- If you want to run with sql query, you can do so but for one at a time
SELECT * FROM workspace.mimic_iv_dataset_bronze.diagnoses_icd LIMIT 10